# Data access: getting back what you stored

Your gateway publishes the building's sensors on MQTT and your `iot/subscriber.py` writes them into InfluxDB **with your own schema** (measurement names, tags, fields, units). Stored data are only useful if you can get them back and turn them into answers: *how much energy did the building use each day? for how many hours was it uncomfortable? did my controller improve anything?*

There are two databases:

| database | run |
|---|---|
| `base_building` | baseline: native EnergyPlus control |
| `controlled_building` | controlled: your BMS closes the loop |

In this notebook you will
1. explore the schema **you** designed,
2. write the InfluxQL (+ pandas) queries for four typical questions,
3. write down your schema so that the next notebook (energy signature) can rely on it.

The helper `query(db, influxql)` is given. It uses the InfluxDB 1.x HTTP API with the standard library only (the `influxdb` Python package is not installed) and returns a pandas DataFrame.

In [ ]:
import base64, json, os, urllib.error, urllib.parse, urllib.request
import numpy as np
import pandas as pd

INFLUX_URL  = os.getenv("INFLUX_URL", "http://influxdb:8086")
INFLUX_USER = os.getenv("INFLUX_USERNAME", "admin")
INFLUX_PASS = os.getenv("INFLUX_PASSWORD", "")

DB_BASE = "base_building"        # baseline run: native EnergyPlus control
DB_CTRL = "controlled_building"  # controlled run: your BMS closes the loop


def query(db, influxql):
    """Run one InfluxQL statement on database `db` and return a pandas DataFrame.

    - SELECT: one row per point; the `time` column is a UTC datetime; tags of a
      GROUP BY are added as columns; several series are concatenated.
    - SHOW ...: the plain table returned by InfluxDB.
    - empty result: an empty DataFrame (check `df.empty`).
    Only read statements work here (HTTP GET).
    """
    params = urllib.parse.urlencode({"db": db, "q": influxql, "epoch": "ms"})
    req = urllib.request.Request(f"{INFLUX_URL}/query?{params}")
    token = base64.b64encode(f"{INFLUX_USER}:{INFLUX_PASS}".encode()).decode()
    req.add_header("Authorization", "Basic " + token)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            body = json.load(resp)
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"InfluxDB answered HTTP {e.code}: {e.read().decode(errors='replace')[:300]}") from None
    except urllib.error.URLError as e:
        raise RuntimeError(f"Cannot reach InfluxDB at {INFLUX_URL}: {e.reason}") from None
    frames = []
    for result in body.get("results", []):
        if "error" in result:
            raise RuntimeError(f"InfluxQL error in database '{db}': {result['error']}")
        for series in result.get("series", []):
            df = pd.DataFrame(series.get("values", []), columns=series["columns"])
            for tag, value in (series.get("tags") or {}).items():
                df[tag] = value
            if "time" in df.columns:
                df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 1. Explore your schema

InfluxDB 1.x is schema-on-write: whatever your `subscriber.py` wrote is what is there. Remember the vocabulary:

- **measurement**: like a table;
- **tag**: indexed string metadata (device, point, run, unit, ...), used in `WHERE` and `GROUP BY`;
- **field**: the value itself (numeric, string, boolean), not indexed;
- **series**: one measurement plus one combination of tag values.

Run the cell below. If a database is empty, run the building first (start your subscriber *before* the building).

In [ ]:
for db in (DB_BASE, DB_CTRL):
    print("=" * 70, "\n", db)
    try:
        measurements = query(db, "SHOW MEASUREMENTS")
    except RuntimeError as err:
        print("  ", err); continue
    if measurements.empty:
        print("   empty database: run the building (with your subscriber running) first"); continue
    for m in measurements["name"]:
        print(f"\n-- measurement: {m}")
        tags = query(db, f'SHOW TAG KEYS FROM "{m}"')   # empty when the measurement has no tags
        print("tag keys  :", tags["tagKey"].tolist() if not tags.empty else [])
        print("field keys:", query(db, f'SHOW FIELD KEYS FROM "{m}"').values.tolist())

In [ ]:
# More exploration. Replace "<measurement>" and "<tag key>" with names from the output above
# (kept as comments so that the notebook runs as it is).
# query(DB_BASE, 'SHOW TAG VALUES FROM "<measurement>" WITH KEY = "<tag key>"')
# query(DB_BASE, 'SHOW SERIES')
# query(DB_BASE, 'SELECT * FROM "<measurement>" LIMIT 5')

## Write down your schema (fill in)

Double-click and edit. Be precise: the other notebooks and your future self will depend on it.

- measurement name(s): *TODO*
- tags (and what values they take): *TODO*
- fields: *TODO*
- how the **run** (baseline / controlled) is stored (database only? a tag?): *TODO*
- the point/quantity that gives the **electric or thermal energy**: name, unit, and is it an **increment per step** (sum it) or a **cumulative meter** (difference of last and first)? How did you check? *TODO*
- the point that gives the **zone temperature** (unit?): *TODO*
- the time stamp: UTC? simulation time? what is the step? *TODO*
- known quirks (duplicates, gaps, strange values): *TODO*

## 2a. Last value of every point

**Goal.** One row per point (device/quantity) with its most recent value and the time stamp of that value.

**Expected output.** A DataFrame with one row per point, e.g. columns `point`, `time`, `last_value` (and the unit if you store it). The number of rows is the number of points your gateway publishes. Hints: `SELECT LAST("<field>") FROM ... GROUP BY "<tag>"`, or `LAST(*)` for all fields.

In [ ]:
# TODO (a): last value of every point of the baseline database
raise NotImplementedError("TODO (a): write the InfluxQL query for the last value of every point")

## 2b. Daily energy of the baseline

**Goal.** Energy used by the building on each day of the baseline run, in kWh.

**First check what you stored.** Look at the raw values of your energy point:
- an **increment per step** (energy of each time step, e.g. `Electricity:Facility` in J per step): the daily energy is the **sum** of the samples of that day;
- a **cumulative meter** (monotonically increasing): the daily energy is **last - first** of the day (and beware of the first day and of resets).

Conversion: 1 kWh = 3.6e6 J. Be careful with the time stamp: EnergyPlus reports at the **end** of a step, so the sample stamped `00:00` belongs to the previous day. You may group on the server (`GROUP BY time(1d)`) or in pandas (`resample("1D")`). If your data could be stored twice (duplicate series), server-side `SUM` would double count.

**Expected output.** One row per day (7 rows for a 7-day simulation) with the energy in kWh, plus a short bar plot.

In [ ]:
# TODO (b): daily energy of the baseline in kWh -> a pandas Series/DataFrame, one row per day
raise NotImplementedError("TODO (b): increment or cumulative? then sum (or last-first) per day and convert to kWh")

## 2c. Hours outside a comfort band

**Goal.** For each day, count the hours in which the zone temperature is **outside** the comfort band [20, 26] degC (below 20 or above 26), in the baseline.

Think about it:
- samples are not necessarily hourly (check your step, and whether you store on change only): you want *hours*, not *number of samples*. A robust way is to bring the series to a regular 1 h grid (`resample("1h").mean()`, then `ffill()` if you have gaps) and count the hours of the grid;
- if you have several zones, decide whether a zone is uncomfortable when *its own* temperature is out of the band (then a zone-hour count) or use one reference zone, and say which.

**Expected output.** One row per day with `hours_outside`. In a 7-day simulation of a building with a heating setpoint of 20 degC and no cooling in winter, expect a few hours at most (or 0).

In [ ]:
BAND = (20.0, 26.0)   # degC

# TODO (c): hours per day outside BAND for the zone temperature of the baseline
raise NotImplementedError("TODO (c): regular 1 h grid, flag the hours outside BAND, count per day")

## 2d. Baseline versus controlled, same period

**Goal.** Put the two runs side by side for the **same period**: daily energy (kWh) and hours outside the comfort band, one column per run, plus the difference (controlled - baseline) and the total saving in %.

Remember that the two runs live in **different databases** (`DB_BASE`, `DB_CTRL`): run the same query twice, then `join` the results on the day. Intersect the periods first: comparing a week with a day means nothing.

**Expected output.** A table indexed by day with the columns for the two runs and the difference; one grouped bar plot for the energy; one sentence on whether the controller saved energy and at which comfort price.

In [ ]:
# TODO (d): daily energy and hours outside the band for both runs, side by side + difference
raise NotImplementedError("TODO (d): run the queries on DB_BASE and DB_CTRL, join on the day, compute the difference")

**Your reading of the result (fill in):** did the controller save energy? at what comfort price? is the comparison fair (same period, same weather)? *TODO*